## Parsing via Docling (isolated, no VLM)

Isolating Docling alone (no VLM) for inspection: importing the stack, reporting the GPU and selecting the SVM lecture as the test document

In [ ]:
import time
import json
import os
import base64
import torch
from openai import OpenAI
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from io import BytesIO

print("PyTorch:", torch.__version__)
print("CUDA verfuegbar:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PDF_PATH = Path(os.getcwd()).parent / "data" / "raw" / "pdfs"
TEST_PRESENTATION = next(PDF_PATH.glob("*svm.pdf"))

TEST_PRESENTATION.name

Running Docling with formula enrichment off and exporting per page Markdown with image placeholders, the raw structural output before any VLM

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling_core.types.doc import ImageRefMode

options = PdfPipelineOptions()
options.do_ocr = False
options.do_table_structure = True
options.do_formula_enrichment = False
options.generate_page_images = True
options.images_scale = 2

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
)

print("Docling starts parsing ...")
start = time.time()
result = converter.convert(TEST_PRESENTATION)
document = result.document
print(f"Finished in {time.time() - start:.1f} seconds.")
print("Pages:", len(document.pages))

pages: dict[int, Image.Image] = {}
anker_texts: dict[int, str] = {}

for page in result.pages:
    pages[page.page_no] = page.image
    anker_texts[page.page_no] = document.export_to_markdown(
        page_no=page.page_no,
         image_mode=ImageRefMode.PLACEHOLDER,
        )

print(f"Seiten-Indizes: {sorted(pages.keys())}")

## Docling output

Displaying each page beside its Docling Markdown, highlighting the un transcribed image and undecoded formula placeholders to expose where pure Docling loses content

In [ ]:
import html as html_lib

def display_pages_and_markdown(page_no: int):
    buffer = BytesIO()
    pages[page_no].save(buffer, format="PNG")
    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    md = html_lib.escape(anker_texts[page_no])

    md = md.replace(
        html_lib.escape("<!-- image -->"),
        '<span style="background:#ffd6d6; color:#a00; font-weight:600;">[IMAGE – nicht transkribiert]</span>',
    ).replace(
        html_lib.escape("<!-- formula-not-decoded -->"),
        '<span style="background:#fff0c2; color:#9a6a00; font-weight:600;">[FORMEL – nicht dekodiert]</span>',
    )

    display(HTML(f'''
    <div style="display: flex; gap: 20px; align-items: flex-start;">
        <img src="data:image/png;base64,{img_b64}" style="width: 50%; border: 1px solid #ccc;"/>
        <div style="width: 50%; overflow-y: auto; max-height: 800px; padding: 10px;">
            <pre style="white-space: pre-wrap; font-size: 13px;">{md}</pre>
        </div>
    </div>
    '''))

for page_no in pages:
    display_pages_and_markdown(page_no)


Drawing Docling's detected bounding boxes per label (formulas in red) on selected slides, to visualise the layout detection and motivate why a VLM stage is needed

In [ ]:
from PIL import ImageDraw, ImageFont
from docling_core.types.doc import DocItemLabel
from pathlib import Path
import os

EXAMPLE_DIR = Path(os.getcwd()).parent / "data" / "eval" / "docling_beispiele"
EXAMPLE_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    DocItemLabel.FORMULA:        "#e6194B",  
    DocItemLabel.PICTURE:        "#4363d8", 
    DocItemLabel.TABLE:          "#911eb4",   
    DocItemLabel.SECTION_HEADER: "#f58231",   
    DocItemLabel.TEXT:           "#3cb44b",  
    DocItemLabel.LIST_ITEM:      "#42d4f4",   
    DocItemLabel.CAPTION:        "#808000",  
}
DEFAULT_COLOR = "#999999"

def _font(size):
    try:
        return ImageFont.truetype("arial.ttf", size)
    except Exception:
        return ImageFont.load_default()
def annotate_page(page_no: int, show_labels: bool = True):
    page = document.pages[page_no]
    img  = pages[page_no].convert("RGB").copy()
    sx   = img.width  / page.size.width
    sy   = img.height / page.size.height
    H_pt = page.size.height
    draw = ImageDraw.Draw(img)
    tag_font = _font(13)

    counts = {}
    for item, _ in document.iterate_items():
        label = getattr(item, "label", None)
        color = PALETTE.get(label, DEFAULT_COLOR)
        name  = str(getattr(label, "value", label))
        for prov in getattr(item, "prov", []):
            if prov.page_no != page_no:
                continue
            bb  = prov.bbox.to_top_left_origin(page_height=H_pt)
            box = [bb.l * sx, bb.t * sy, bb.r * sx, bb.b * sy]
            draw.rectangle(box, outline=color, width=3)
            if show_labels:
                tb = draw.textbbox((0, 0), name, font=tag_font)
                tw, th = tb[2] - tb[0], tb[3] - tb[1]
                tx, ty = box[0], max(0, box[1] - th - 4)
                draw.rectangle([tx, ty, tx + tw + 6, ty + th + 4], fill=color)
                draw.text((tx + 3, ty + 2), name, fill="white", font=tag_font)
            counts[label] = counts.get(label, 0) + 1

    f = _font(max(14, img.width // 70))
    present = list(counts.items())
    x0, y0, lh = img.width - 230, 10, 22
    draw.rectangle([x0 - 10, y0 - 6, img.width - 8, y0 + lh * len(present) + 4],
                   fill="white", outline="#333", width=1)
    for i, (lbl, c) in enumerate(present):
        y = y0 + i * lh
        col = PALETTE.get(lbl, DEFAULT_COLOR)
        draw.rectangle([x0, y + 3, x0 + 16, y + 16], fill=col)
        draw.text((x0 + 24, y), f"{getattr(lbl,'value',lbl)} ({c})", fill="#222", font=f)

    out = EXAMPLE_DIR / f"svm_folie_{page_no}_docling_boxes.png"
    img.save(out)
    print(f"Folie {page_no:>2}: FORMULA={counts.get(DocItemLabel.FORMULA, 0)} | {out.name}")
    return img

from IPython.display import display
for page_no in [8, 10, 11]:
    display(annotate_page(page_no))